# AML Synthetic Data Generator -- Master Data & Transactions
---
**Purpose**: Generate realistic, schema-compliant synthetic data for Anti-Money Laundering model training.
**Outputs**: `customers.csv`, `accounts.csv`, `wallets.csv`, `devices.csv`, `transactions_base.csv`, `config.json`
**Primary Keys**: `customer_cif` -> `account_number` / `wallet_id` -> `transaction_id`


## 1 -- Environment Setup


In [ ]:
import random, string, hashlib, uuid, json, os, math
from datetime import datetime, timedelta, date
from collections import defaultdict, Counter
import csv
import warnings
warnings.filterwarnings('ignore')

random.seed(42)

OUTPUT_DIR = 'aml_synthetic_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output directory: {os.path.abspath(OUTPUT_DIR)}')


## 2 -- Configuration


In [ ]:
# ============================================================
# MASTER CONFIGURATION
# ============================================================
CONFIG = {
    "num_customers_individual": 4000,
    "num_customers_entity": 1000,
    "num_accounts_per_customer_range": [1, 3],
    "num_wallets_fraction": 0.30,
    "num_transactions_per_account_range": [10, 120],
    "date_range_start": "2024-01-01",
    "date_range_end": "2025-03-31",
    "primary_currency": "INR",
    "fx_currencies": ["USD", "GBP", "EUR", "AED", "SGD"],
    "fx_probability": 0.03,
    "risk_weights": {"Low": 0.60, "Medium": 0.30, "High": 0.10},
    "pep_probability": 0.02,
    "hni_probability": 0.05,
    "minor_probability": 0.03,
    "income_bands": {
        "Low":    [100000, 500000],
        "Medium": [500001, 2500000],
        "High":   [2500001, 10000000],
        "HNI":    [10000001, 100000000]
    },
    "wallet_kyc_categories": {
        "Minimum KYC": {
            "per_txn": 10000, "daily": 10000, "monthly": 10000,
            "annual": 120000, "max_balance": 10000
        },
        "Full KYC": {
            "per_txn": 200000, "daily": 200000, "monthly": 200000,
            "annual": 2400000, "max_balance": 200000
        }
    },
    "bank_ifsc_prefixes": [
        "SBIN0", "HDFC0", "ICIC0", "UTIB0", "PUNB0",
        "BARB0", "CNRB0", "KKBK0", "IOBA0", "BKID0"
    ],
    "mcc_codes": {
        "5411": "Grocery Stores", "5541": "Service Stations",
        "5812": "Restaurants", "5912": "Drug Stores",
        "4814": "Telecom Services", "6012": "Financial Institutions",
        "7011": "Hotels/Motels", "5691": "Clothing Stores",
        "5045": "Computers/Peripherals", "8062": "Hospitals",
        "4899": "Cable/Utilities", "5999": "Miscellaneous Retail",
        "6051": "Quasi Cash - Money Orders", "6211": "Security Brokers/Dealers",
        "7995": "Gambling/Betting"
    },
    "bank_channels": [
        "NEFT", "RTGS", "IMPS", "UPI", "Branch Cash",
        "ATM", "Cheque", "Internet Banking", "Mobile Banking",
        "POS", "Demand Draft"
    ],
    "ppi_channels": ["UPI", "QR Scan", "In-App Transfer", "NFC Tap", "Online"],
    "auth_methods": ["OTP", "PIN", "Biometric", "MPIN", "Password"],
    "browsers": [
        "Chrome/124.0", "Safari/17.4", "Firefox/125.0",
        "Edge/124.0", "SamsungBrowser/24.0"
    ],
    "app_versions": ["v8.1.2", "v8.2.0", "v9.0.1", "v9.1.0", "v7.5.3"],
    "indian_states": [
        "Maharashtra", "Delhi", "Karnataka", "Tamil Nadu", "Telangana",
        "Gujarat", "Rajasthan", "Uttar Pradesh", "West Bengal", "Kerala",
        "Madhya Pradesh", "Punjab", "Haryana", "Bihar", "Odisha"
    ],
    "cities_by_state": {
        "Maharashtra":   ["Mumbai", "Pune", "Nagpur", "Nashik"],
        "Delhi":         ["New Delhi", "Dwarka", "Rohini", "Saket"],
        "Karnataka":     ["Bengaluru", "Mysuru", "Hubli", "Mangaluru"],
        "Tamil Nadu":    ["Chennai", "Coimbatore", "Madurai", "Salem"],
        "Telangana":     ["Hyderabad", "Warangal", "Nizamabad", "Karimnagar"],
        "Gujarat":       ["Ahmedabad", "Surat", "Vadodara", "Rajkot"],
        "Rajasthan":     ["Jaipur", "Jodhpur", "Udaipur", "Kota"],
        "Uttar Pradesh": ["Lucknow", "Noida", "Agra", "Varanasi"],
        "West Bengal":   ["Kolkata", "Howrah", "Siliguri", "Durgapur"],
        "Kerala":        ["Kochi", "Thiruvananthapuram", "Kozhikode", "Thrissur"],
        "Madhya Pradesh":["Bhopal", "Indore", "Gwalior", "Jabalpur"],
        "Punjab":        ["Chandigarh", "Ludhiana", "Amritsar", "Jalandhar"],
        "Haryana":       ["Gurugram", "Faridabad", "Karnal", "Panipat"],
        "Bihar":         ["Patna", "Gaya", "Muzaffarpur", "Bhagalpur"],
        "Odisha":        ["Bhubaneswar", "Cuttack", "Rourkela", "Puri"]
    },
    "occupations_individual": [
        "Salaried - IT", "Salaried - Banking", "Salaried - Government",
        "Self-Employed - Retail", "Self-Employed - Professional",
        "Business Owner", "Freelancer", "Student", "Retired",
        "Agriculture", "Doctor", "Lawyer", "Consultant"
    ],
    "entity_types": [
        "Private Limited", "Public Limited", "LLP",
        "Partnership", "Trust", "Society", "Sole Proprietorship",
        "HUF", "Government Body"
    ],
    "entity_industries": [
        "IT Services", "Manufacturing", "Trading", "Real Estate",
        "Financial Services", "Healthcare", "Education", "Hospitality",
        "Logistics", "Textiles", "Pharmaceuticals", "Agriculture",
        "Construction", "Retail", "Import/Export"
    ]
}

with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)
print("Configuration saved")



## 3 -- Helper Functions


In [ ]:
# -- Realistic ID generators --

def generate_cif():
    # 12-digit CIF like major Indian banks (SBI, HDFC)
    return str(random.randint(100000000000, 999999999999))

def generate_account_number(ifsc_prefix):
    # Realistic Indian bank account numbers by bank
    bank = ifsc_prefix[:4]
    if bank == "SBIN":
        return str(random.randint(10000000000, 99999999999))          # 11 digits
    elif bank == "HDFC":
        return str(random.randint(10000000000000, 99999999999999))    # 14 digits
    elif bank == "ICIC":
        return str(random.randint(100000000000, 999999999999))        # 12 digits
    elif bank == "UTIB":  # Axis
        return str(random.randint(9100000000000, 9299999999999))      # 13 digits starting 9[12]
    elif bank == "PUNB":
        return str(random.randint(1000000000000000, 9999999999999999))  # 16 digits
    else:
        length = random.choice([11, 12, 13, 14])
        return str(random.randint(10**(length-1), 10**length - 1))

def generate_ifsc():
    prefix = random.choice(CONFIG["bank_ifsc_prefixes"])
    return prefix + str(random.randint(10000, 99999))

def generate_pan(is_entity=False):
    # PAN format: AAAPL1234C - 4th char encodes entity type
    first3 = ''.join(random.choices(string.ascii_uppercase, k=3))
    fourth = random.choice(["C", "H", "F", "A", "T", "B", "L", "J", "G"]) if is_entity else "P"
    fifth = random.choice(string.ascii_uppercase)
    digits = ''.join(random.choices(string.digits, k=4))
    last = random.choice(string.ascii_uppercase)
    return first3 + fourth + fifth + digits + last

def generate_aadhaar():
    # 12-digit Aadhaar starting with 2-9
    first = str(random.randint(2, 9))
    rest = ''.join(random.choices(string.digits, k=11))
    return first + rest

def generate_mobile():
    # Indian mobile: starts with 6-9, 10 digits
    return str(random.choice([6,7,8,9])) + ''.join(random.choices(string.digits, k=9))

def generate_email(name):
    domains = ["gmail.com", "yahoo.co.in", "outlook.com", "rediffmail.com", "hotmail.com"]
    clean = name.lower().replace(" ", ".").replace("'", "")
    return f"{clean}{random.randint(1,999)}@{random.choice(domains)}"

def generate_vpa(name):
    # UPI VPA like name@bankhandle
    handles = ["oksbi", "okaxis", "okicici", "okhdfcbank", "ybl", "paytm", "ibl", "upi"]
    clean = name.lower().replace(" ", "")[:10]
    return f"{clean}{random.randint(1,99)}@{random.choice(handles)}"

def generate_wallet_id():
    return "WLT" + ''.join(random.choices(string.digits, k=12))

def generate_merchant_id():
    return "MID" + ''.join(random.choices(string.ascii_uppercase + string.digits, k=12))

def generate_device_id():
    return hashlib.sha256(uuid.uuid4().bytes).hexdigest()[:32]

def generate_ip():
    # Indian IP ranges (rough approximation)
    first_octet = random.choice([49, 59, 103, 106, 117, 122, 157, 182, 203])
    return f"{first_octet}.{random.randint(0,255)}.{random.randint(0,255)}.{random.randint(1,254)}"

def generate_gps(city, state):
    # Approximate GPS for major Indian cities
    coords = {
        "Mumbai": (19.076, 72.878), "Pune": (18.520, 73.856),
        "New Delhi": (28.614, 77.209), "Bengaluru": (12.972, 77.594),
        "Chennai": (13.083, 80.271), "Hyderabad": (17.385, 78.487),
        "Ahmedabad": (23.023, 72.571), "Kolkata": (22.573, 88.364),
        "Jaipur": (26.912, 75.787), "Lucknow": (26.847, 80.947),
        "Kochi": (9.932, 76.267), "Bhopal": (23.260, 77.413),
        "Chandigarh": (30.734, 76.779), "Patna": (25.611, 85.144),
        "Bhubaneswar": (20.297, 85.825),
    }
    base = coords.get(city, (20.5 + random.uniform(-5, 5), 78.0 + random.uniform(-5, 5)))
    return (round(base[0] + random.uniform(-0.05, 0.05), 6),
            round(base[1] + random.uniform(-0.05, 0.05), 6))

def random_date(start_str, end_str):
    start = datetime.strptime(start_str, "%Y-%m-%d")
    end = datetime.strptime(end_str, "%Y-%m-%d")
    delta = (end - start).days
    if delta <= 0:
        return start
    return start + timedelta(days=random.randint(0, delta))

def weighted_choice(weight_dict):
    items = list(weight_dict.keys())
    weights = list(weight_dict.values())
    return random.choices(items, weights=weights, k=1)[0]

# -- Name generators --

FIRST_NAMES_M = ["Rahul", "Amit", "Suresh", "Vikram", "Arjun", "Rohan", "Kiran", "Deepak",
                  "Manoj", "Rajesh", "Sanjay", "Anil", "Pradeep", "Nikhil", "Varun",
                  "Aditya", "Akash", "Harsh", "Yash", "Gaurav", "Sachin", "Pranav",
                  "Ankit", "Ravi", "Sandeep", "Mohit", "Tushar", "Vivek", "Ashish", "Naveen"]

FIRST_NAMES_F = ["Priya", "Anita", "Sunita", "Kavita", "Neha", "Pooja", "Swati", "Meera",
                  "Divya", "Sneha", "Ritu", "Pallavi", "Shruti", "Aditi", "Anjali",
                  "Nisha", "Rekha", "Sonal", "Tanvi", "Bhavna", "Komal", "Jyoti",
                  "Asha", "Geeta", "Lata", "Mamta", "Rashmi", "Sapna", "Shweta", "Vandana"]

LAST_NAMES = ["Sharma", "Verma", "Patel", "Gupta", "Singh", "Kumar", "Reddy", "Nair",
              "Iyer", "Joshi", "Deshmukh", "Rao", "Pillai", "Mehta", "Shah",
              "Agarwal", "Mishra", "Bhat", "Yadav", "Chauhan", "Tiwari", "Pandey",
              "Malhotra", "Kapoor", "Bose", "Chatterjee", "Mukherjee", "Das", "Banerjee", "Sen"]

ENTITY_PREFIXES = ["Shri", "Om", "Bharat", "National", "Premier", "Golden", "Silver",
                   "Diamond", "Royal", "Star", "Global", "United", "Pioneer", "Excel", "Apex"]
ENTITY_SUFFIXES = ["Enterprises", "Trading Co", "Industries", "Solutions", "Services",
                   "Exports", "Imports", "Associates", "Group", "Holdings",
                   "Infra", "Ventures", "Technologies", "Logistics", "Capital"]

def random_individual_name():
    if random.random() < 0.5:
        return random.choice(FIRST_NAMES_M) + " " + random.choice(LAST_NAMES)
    return random.choice(FIRST_NAMES_F) + " " + random.choice(LAST_NAMES)

def random_entity_name():
    return random.choice(ENTITY_PREFIXES) + " " + random.choice(ENTITY_SUFFIXES)

def random_father_name():
    return random.choice(FIRST_NAMES_M) + " " + random.choice(LAST_NAMES)

MERCHANT_NAMES = [
    "Reliance Fresh", "Big Bazaar", "DMart", "More Supermarket",
    "Swiggy", "Zomato", "Amazon India", "Flipkart", "Myntra",
    "BookMyShow", "MakeMyTrip", "IRCTC", "Uber India", "Ola Cabs",
    "Jio Payments", "Airtel Store", "Apollo Pharmacy", "MedPlus",
    "Croma Electronics", "Vijay Sales", "Tanishq", "Titan Eye+",
    "PVR Cinemas", "Decathlon India", "Nykaa", "Lenskart",
    "Urban Company", "PhonePe Merchant", "Google Pay Merchant", "Paytm Mall"
]

print("Helpers loaded")



## 4 -- Generate Customer Master Data


In [ ]:
customers = []
cif_set = set()

def make_individual_customer(idx):
    cif = generate_cif()
    while cif in cif_set:
        cif = generate_cif()
    cif_set.add(cif)

    state = random.choice(CONFIG["indian_states"])
    city = random.choice(CONFIG["cities_by_state"][state])
    gps = generate_gps(city, state)
    name = random_individual_name()

    is_pep = random.random() < CONFIG["pep_probability"]
    is_hni = random.random() < CONFIG["hni_probability"]
    is_minor = random.random() < CONFIG["minor_probability"]
    risk = weighted_choice(CONFIG["risk_weights"])
    if is_pep:
        risk = "High"

    if is_hni:
        income_band = CONFIG["income_bands"]["HNI"]
    elif risk == "High":
        income_band = CONFIG["income_bands"][random.choice(["Medium", "High"])]
    else:
        income_band = CONFIG["income_bands"][risk]
    annual_income = random.randint(*income_band)

    dob = random_date("1960-01-01", "2006-12-31") if not is_minor else random_date("2008-01-01", "2015-12-31")
    cif_creation = random_date("2015-01-01", "2024-06-30")
    kyc_date = random_date(cif_creation.strftime("%Y-%m-%d"), "2025-03-15")
    occupation = random.choice(CONFIG["occupations_individual"])

    return {
        "customer_cif": cif,
        "customer_name": name,
        "customer_type": "Individual",
        "customer_entity_type": "Individual",
        "date_of_birth": dob.strftime("%d-%m-%Y"),
        "father_spouse_name": random_father_name(),
        "nationality": "Indian",
        "citizenship": "Indian",
        "residency": random.choices(["Resident", "NRI"], weights=[0.95, 0.05])[0],
        "tax_residency": random.choices(["India", "Other"], weights=[0.96, 0.04])[0],
        "pan": generate_pan(is_entity=False),
        "aadhaar": generate_aadhaar(),
        "identification_doc_no": "AADHAAR-" + generate_aadhaar()[:4] + "XXXX" + generate_aadhaar()[-4:],
        "mobile_number": generate_mobile(),
        "email_id": generate_email(name),
        "address_individual": f"{random.randint(1,500)}, {random.choice(['MG Road','Station Road','NH Highway','Park Street','Ring Road','Gandhi Nagar','Nehru Place'])}, {city}, {state} - {random.randint(100000,999999)}",
        "state": state,
        "city": city,
        "address_lat": gps[0],
        "address_lon": gps[1],
        "occupation_industry": occupation,
        "annual_income": annual_income,
        "professional_experience_years": 0 if is_minor else random.randint(0, 40),
        "source_of_funds": random.choice(["Salary", "Business Income", "Investment Returns", "Rental Income", "Family Support", "Pension", "Agriculture Income"]),
        "customer_risk_score": risk,
        "pep_flag": "Y" if is_pep else "N",
        "hni_flag": "Y" if is_hni else "N",
        "minor_flag": "Y" if is_minor else "N",
        "non_face_to_face_flag": random.choices(["Y", "N"], weights=[0.35, 0.65])[0],
        "vkyc_flag": random.choices(["Y", "N"], weights=[0.25, 0.75])[0],
        "cif_creation_date": cif_creation.strftime("%d-%m-%Y"),
        "kyc_update_date": kyc_date.strftime("%d-%m-%Y"),
        "date_of_incorporation": "",
        "place_of_incorporation": "",
        "beneficial_owner_types": "",
        "passive_nfe": "",
        "address_registered_office": "",
        "address_place_of_business": "",
        "address_beneficial_owners": "",
        "entity_identification_doc_no": "",
        "cif_beneficial_owners": "",
        "name_beneficial_owners": "",
    }

def make_entity_customer(idx):
    cif = generate_cif()
    while cif in cif_set:
        cif = generate_cif()
    cif_set.add(cif)

    state = random.choice(CONFIG["indian_states"])
    city = random.choice(CONFIG["cities_by_state"][state])
    gps = generate_gps(city, state)
    entity_name = random_entity_name()
    entity_type = random.choice(CONFIG["entity_types"])
    industry = random.choice(CONFIG["entity_industries"])
    risk = weighted_choice(CONFIG["risk_weights"])
    inc_date = random_date("1990-01-01", "2024-01-01")
    cif_creation = random_date(inc_date.strftime("%Y-%m-%d"), "2024-06-30")
    if cif_creation < inc_date:
        cif_creation = inc_date + timedelta(days=random.randint(1, 365))
    kyc_date = random_date(cif_creation.strftime("%Y-%m-%d"), "2025-03-15")

    ubo_name = random_individual_name()
    income = random.randint(1000000, 500000000)

    return {
        "customer_cif": cif,
        "customer_name": entity_name,
        "customer_type": "Non-Individual",
        "customer_entity_type": entity_type,
        "date_of_birth": "",
        "father_spouse_name": "",
        "nationality": "Indian",
        "citizenship": "",
        "residency": "Resident",
        "tax_residency": "India",
        "pan": generate_pan(is_entity=True),
        "aadhaar": "",
        "identification_doc_no": "",
        "mobile_number": generate_mobile(),
        "email_id": generate_email(entity_name.split()[0]),
        "address_individual": "",
        "state": state,
        "city": city,
        "address_lat": gps[0],
        "address_lon": gps[1],
        "occupation_industry": industry,
        "annual_income": income,
        "professional_experience_years": "",
        "source_of_funds": random.choice(["Business Revenue", "Investment", "Government Grant", "Donations", "Trading Profits"]),
        "customer_risk_score": risk,
        "pep_flag": "N",
        "hni_flag": "N",
        "minor_flag": "N",
        "non_face_to_face_flag": random.choices(["Y", "N"], weights=[0.20, 0.80])[0],
        "vkyc_flag": "N",
        "cif_creation_date": cif_creation.strftime("%d-%m-%Y"),
        "kyc_update_date": kyc_date.strftime("%d-%m-%Y"),
        "date_of_incorporation": inc_date.strftime("%d-%m-%Y"),
        "place_of_incorporation": f"{city}, {state}",
        "beneficial_owner_types": random.choice(["Shareholding > 25%", "Control through other means", "Senior Managing Official"]),
        "passive_nfe": random.choices(["Y", "N"], weights=[0.15, 0.85])[0],
        "address_registered_office": f"{random.randint(1,200)} Corporate Park, {city}, {state} - {random.randint(100000,999999)}",
        "address_place_of_business": f"Plot {random.randint(1,500)}, Industrial Area, {city}, {state}",
        "address_beneficial_owners": f"{random.randint(1,999)}, Residential Colony, {city}, {state}",
        "entity_identification_doc_no": f"U{random.randint(10000,99999)}{random.choice(string.ascii_uppercase)}{random.choice(string.ascii_uppercase)}{random.randint(1990,2024)}PTC{random.randint(100000,999999)}",
        "cif_beneficial_owners": generate_cif(),
        "name_beneficial_owners": ubo_name,
    }

print("Generating individual customers...")
for i in range(CONFIG["num_customers_individual"]):
    customers.append(make_individual_customer(i))

print("Generating entity customers...")
for i in range(CONFIG["num_customers_entity"]):
    customers.append(make_entity_customer(i))

random.shuffle(customers)
print(f"Total customers generated: {len(customers)}")



## 5 -- Generate Accounts


In [ ]:
accounts = []
acct_number_set = set()
cif_to_accounts = defaultdict(list)

ACCOUNT_CATEGORIES = ["Savings", "Current", "Salary", "NRE", "NRO", "Fixed Deposit", "Recurring Deposit"]
ACCOUNT_TYPES = ["Regular", "Premium", "Basic", "Corporate", "BSBD"]

for cust in customers:
    cif = cust["customer_cif"]
    n_accts = random.randint(*CONFIG["num_accounts_per_customer_range"])
    ifsc = generate_ifsc()

    for j in range(n_accts):
        acct_num = generate_account_number(ifsc[:5])
        while acct_num in acct_number_set:
            acct_num = generate_account_number(ifsc[:5])
        acct_number_set.add(acct_num)

        cif_date = datetime.strptime(cust["cif_creation_date"], "%d-%m-%Y")
        start_bound = max(datetime.strptime("2015-01-01", "%Y-%m-%d"), cif_date)
        open_date = random_date(start_bound.strftime("%Y-%m-%d"), "2024-12-31")

        is_dormant = random.random() < 0.05
        status = "Dormant" if is_dormant else random.choices(
            ["Active", "Frozen", "Closed"], weights=[0.90, 0.03, 0.02]
        )[0]
        inoperative_date = ""
        if is_dormant:
            inoperative_date = random_date(open_date.strftime("%Y-%m-%d"), "2025-03-15").strftime("%d-%m-%Y")

        cat = random.choice(ACCOUNT_CATEGORIES)
        if cust["customer_type"] == "Non-Individual":
            cat = random.choice(["Current", "Fixed Deposit"])

        acct = {
            "account_number": acct_num,
            "customer_cif": cif,
            "customer_branch_ifsc": ifsc,
            "account_category": cat,
            "account_type": random.choice(ACCOUNT_TYPES),
            "account_status": status,
            "account_opening_date": open_date.strftime("%d-%m-%Y"),
            "inoperative_status_date": inoperative_date,
            "credit_summation_period": 0,
            "debit_summation_period": 0,
        }
        accounts.append(acct)
        cif_to_accounts[cif].append(acct_num)

print(f"Total accounts generated: {len(accounts)}")
acct_lookup = {a["account_number"]: a for a in accounts}



## 6 -- Generate Wallets


In [ ]:
wallets = []
wallet_id_set = set()
cif_to_wallet = {}

individual_cifs = [c["customer_cif"] for c in customers if c["customer_type"] == "Individual"]
wallet_cifs = random.sample(individual_cifs, int(len(individual_cifs) * CONFIG["num_wallets_fraction"]))

for cif in wallet_cifs:
    wid = generate_wallet_id()
    while wid in wallet_id_set:
        wid = generate_wallet_id()
    wallet_id_set.add(wid)

    kyc_cat = random.choices(["Minimum KYC", "Full KYC"], weights=[0.40, 0.60])[0]
    limits = CONFIG["wallet_kyc_categories"][kyc_cat]

    linked_accts = cif_to_accounts.get(cif, [])
    linked_acct = linked_accts[0] if linked_accts else ""

    cust = next(c for c in customers if c["customer_cif"] == cif)
    cif_year = cust["cif_creation_date"].split("-")[-1]
    open_date = random_date(f"{cif_year}-01-01", "2025-01-31")

    w = {
        "wallet_id": wid,
        "customer_cif": cif,
        "wallet_kyc_category": kyc_cat,
        "wallet_status": random.choices(["Active", "Suspended", "Closed"], weights=[0.90, 0.05, 0.05])[0],
        "wallet_opening_date": open_date.strftime("%d-%m-%Y"),
        "escrow_account_linked": f"ESC{random.randint(10**12, 10**13-1)}",
        "per_txn_limit": limits["per_txn"],
        "daily_txn_limit": limits["daily"],
        "monthly_txn_limit": limits["monthly"],
        "annual_txn_limit": limits["annual"],
        "max_balance_limit": limits["max_balance"],
        "linked_bank_account": linked_acct,
    }
    wallets.append(w)
    cif_to_wallet[cif] = wid

print(f"Total wallets generated: {len(wallets)}")



## 7 -- Generate Device Profiles


In [ ]:
devices = []
cif_to_devices = defaultdict(list)

for cust in customers:
    cif = cust["customer_cif"]
    n_devices = random.choices([1, 2, 3], weights=[0.60, 0.30, 0.10])[0]
    for _ in range(n_devices):
        dev = {
            "customer_cif": cif,
            "device_id": generate_device_id(),
            "ip_address": generate_ip(),
            "geo_city": cust["city"],
            "geo_country": "IN",
            "gps_lat": cust["address_lat"] + random.uniform(-0.01, 0.01),
            "gps_lon": cust["address_lon"] + random.uniform(-0.01, 0.01),
            "browser_app_info": random.choice(CONFIG["browsers"] + CONFIG["app_versions"]),
            "vpn_flag": random.choices(["Y", "N"], weights=[0.03, 0.97])[0],
            "emulator_flag": random.choices(["Y", "N"], weights=[0.01, 0.99])[0],
        }
        devices.append(dev)
        cif_to_devices[cif].append(dev)

print(f"Total device profiles generated: {len(devices)}")



## 8 -- Generate Base Transactions
These are **normal** (non-typology) transactions. The typology injector notebook will add AML scenarios on top.


In [ ]:
transactions = []
txn_id_set = set()
used_timestamps = defaultdict(set)  # per account: set of (date, time)

def generate_txn_id():
    return "TXN" + ''.join(random.choices(string.ascii_uppercase + string.digits, k=16))

def unique_timestamp(account_number, txn_date):
    # Ensure no two txns from same account at the same second
    for _ in range(86400):
        h = random.randint(6, 23)
        m = random.randint(0, 59)
        s = random.randint(0, 59)
        ts = f"{h:02d}:{m:02d}:{s:02d}"
        key = (txn_date, ts)
        if key not in used_timestamps[account_number]:
            used_timestamps[account_number].add(key)
            return ts
    return f"{random.randint(0,23):02d}:{random.randint(0,59):02d}:{random.randint(0,59):02d}"

active_accounts = [a for a in accounts if a["account_status"] in ("Active", "Dormant")]
all_acct_numbers = [a["account_number"] for a in active_accounts]

print("Generating base transactions (this may take a minute)...")
for idx, acct in enumerate(active_accounts):
    if idx % 2000 == 0:
        print(f"  Processing account {idx+1}/{len(active_accounts)}...")

    acct_num = acct["account_number"]
    cif = acct["customer_cif"]
    cust = next((c for c in customers if c["customer_cif"] == cif), None)
    if cust is None:
        continue

    n_txns = random.randint(*CONFIG["num_transactions_per_account_range"])
    if acct["account_status"] == "Dormant":
        n_txns = random.randint(0, 5)

    income = cust["annual_income"] if isinstance(cust["annual_income"], int) else 500000
    monthly_budget = income / 12
    dev_list = cif_to_devices.get(cif, [])
    wallet_id = cif_to_wallet.get(cif, "")

    credit_sum = 0
    debit_sum = 0

    for _ in range(n_txns):
        tid = generate_txn_id()
        while tid in txn_id_set:
            tid = generate_txn_id()
        txn_id_set.add(tid)

        txn_date_obj = random_date(CONFIG["date_range_start"], CONFIG["date_range_end"])
        txn_date = txn_date_obj.strftime("%d-%m-%Y")
        txn_time = unique_timestamp(acct_num, txn_date)

        is_fx = random.random() < CONFIG["fx_probability"]
        currency = random.choice(CONFIG["fx_currencies"]) if is_fx else CONFIG["primary_currency"]

        txn_direction = random.choices(["Dr", "Cr"], weights=[0.55, 0.45])[0]

        # Amount distribution
        r = random.random()
        if r < 0.50:
            amount = round(random.uniform(50, 5000), 2)
        elif r < 0.80:
            amount = round(random.uniform(5000, 50000), 2)
        elif r < 0.95:
            amount = round(random.uniform(50000, 200000), 2)
        else:
            amount = round(random.uniform(200000, min(monthly_budget * 2, 5000000)), 2)

        channel = random.choice(CONFIG["bank_channels"])
        is_cash = "Y" if channel in ("Branch Cash", "ATM") else "N"
        is_ppi = random.random() < 0.15 and wallet_id != ""

        ppi_channel = random.choice(CONFIG["ppi_channels"]) if is_ppi else ""
        ppi_txn_type = random.choice(["P2P", "P2M", "Bill Pay", "Recharge", "Cash Out"]) if is_ppi else ""

        # Counterparty
        cp_acct = random.choice(all_acct_numbers)
        while cp_acct == acct_num:
            cp_acct = random.choice(all_acct_numbers)
        cp_info = acct_lookup.get(cp_acct, {})
        cp_ifsc = cp_info.get("customer_branch_ifsc", generate_ifsc())
        cp_cif = cp_info.get("customer_cif", "")
        cp_cust = next((c for c in customers if c["customer_cif"] == cp_cif), None)
        cp_name = cp_cust["customer_name"] if cp_cust else random_individual_name()

        # Merchant (for P2M / POS / Online)
        has_merchant = channel in ("POS", "UPI") or (is_ppi and ppi_txn_type == "P2M") or random.random() < 0.2
        mcc_code = random.choice(list(CONFIG["mcc_codes"].keys())) if has_merchant else ""
        merchant_name = random.choice(MERCHANT_NAMES) if has_merchant else ""
        merchant_id = generate_merchant_id() if has_merchant else ""
        merchant_loc = f"{cust['city']}, {cust['state']}" if has_merchant else ""

        # Wallet balances
        wallet_before = round(random.uniform(0, 200000), 2) if is_ppi else ""
        if is_ppi and isinstance(wallet_before, float):
            if txn_direction == "Dr":
                wallet_after = round(max(0, wallet_before - amount), 2)
            else:
                wallet_after = round(wallet_before + amount, 2)
        else:
            wallet_after = ""

        # Device
        dev = random.choice(dev_list) if dev_list else None
        session_id = hashlib.md5(f"{tid}{txn_date}".encode()).hexdigest()[:24]
        status = random.choices(["Success", "Failed", "Pending", "Reversed"], weights=[0.92, 0.04, 0.02, 0.02])[0]

        if txn_direction == "Cr":
            credit_sum += amount
        else:
            debit_sum += amount

        sender_country = "IN"
        receiver_country = "IN"
        if is_fx:
            if txn_direction == "Cr":
                sender_country = random.choice(["US", "GB", "AE", "SG", "DE"])
            else:
                receiver_country = random.choice(["US", "GB", "AE", "SG", "DE"])

        txn = {
            "transaction_id": tid,
            "timestamp": txn_time,
            "datestamp": txn_date,
            "transaction_amount": amount,
            "currency": currency,
            "transaction_type_dr_cr": txn_direction,
            "transaction_mode_channel_bank": channel,
            "cash_flag": is_cash,
            "transaction_type_ppi": ppi_txn_type,
            "transaction_mode_channel_ppi": ppi_channel,
            "transaction_status": status,
            "wallet_balance_before": wallet_before,
            "wallet_balance_after": wallet_after,
            "source_of_funds_wallet": random.choice(["Bank Transfer", "UPI", "Debit Card", "Credit Card", "Net Banking"]) if is_ppi else "",
            "load_instrument_type": random.choice(["Debit Card", "Net Banking", "UPI", "Credit Card"]) if is_ppi else "",
            "load_source_masked": f"XXXX{random.randint(1000,9999)}" if is_ppi else "",
            "beneficiary_wallet_vpa": generate_vpa(cp_name) if (is_ppi or channel == "UPI") else "",
            "merchant_id": merchant_id,
            "merchant_name": merchant_name,
            "mcc_code": mcc_code,
            "merchant_location": merchant_loc,
            "refund_chargeback_flag": random.choices(["Y", "N"], weights=[0.02, 0.98])[0],
            "customer_account_number": acct_num,
            "customer_cif": cif,
            "counterparty_account_number": cp_acct,
            "counterparty_ifsc_swift": cp_ifsc,
            "customer_name": cust["customer_name"],
            "counterparty_name": cp_name,
            "sender_country_code": sender_country,
            "receiver_country_code": receiver_country,
            "device_id": dev["device_id"] if dev else "",
            "ip_address": dev["ip_address"] if dev else "",
            "geo_city_country": f"{dev['geo_city']}, {dev['geo_country']}" if dev else "",
            "gps_lat": round(dev["gps_lat"], 6) if dev else "",
            "gps_lon": round(dev["gps_lon"], 6) if dev else "",
            "browser_app_info": dev["browser_app_info"] if dev else "",
            "session_id": session_id,
            "auth_method": random.choice(CONFIG["auth_methods"]),
            "vpn_flag": dev["vpn_flag"] if dev else "N",
            "emulator_flag": dev["emulator_flag"] if dev else "N",
            "wallet_id": wallet_id if is_ppi else "",
            "is_aml": 0,
            "aml_typology": "",
            "typology_group_id": "",
        }
        transactions.append(txn)

    # Update account summations
    acct["credit_summation_period"] = round(credit_sum, 2)
    acct["debit_summation_period"] = round(debit_sum, 2)

print(f"Total base transactions generated: {len(transactions)}")



## 9 -- Export All Tables


In [ ]:
def save_csv(data, filename):
    if not data:
        print(f"  WARNING {filename}: empty, skipping")
        return
    filepath = os.path.join(OUTPUT_DIR, filename)
    keys = data[0].keys()
    with open(filepath, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(data)
    print(f"  OK {filename}: {len(data):,} rows, {len(keys)} columns")

print("Saving datasets...")
save_csv(customers, "customers.csv")
save_csv(accounts, "accounts.csv")
save_csv(wallets, "wallets.csv")
save_csv(devices, "devices.csv")
save_csv(transactions, "transactions_base.csv")

print(f"\nAll files saved to: {os.path.abspath(OUTPUT_DIR)}/")
print(f"\n-- Summary --")
print(f"  Customers:    {len(customers):>7,}")
print(f"  Accounts:     {len(accounts):>7,}")
print(f"  Wallets:      {len(wallets):>7,}")
print(f"  Devices:      {len(devices):>7,}")
print(f"  Transactions: {len(transactions):>7,}")



## 10 -- Quick Validation


In [ ]:
cif_from_customers = {c["customer_cif"] for c in customers}
cif_from_accounts  = {a["customer_cif"] for a in accounts}
cif_from_wallets   = {w["customer_cif"] for w in wallets}
cif_from_devices   = {d["customer_cif"] for d in devices}

print("Referential integrity checks:")
print(f"  All account CIFs in customers? {cif_from_accounts.issubset(cif_from_customers)}")
print(f"  All wallet CIFs in customers?  {cif_from_wallets.issubset(cif_from_customers)}")
print(f"  All device CIFs in customers?  {cif_from_devices.issubset(cif_from_customers)}")

acct_nums = [a["account_number"] for a in accounts]
print(f"  Unique account numbers?        {len(acct_nums) == len(set(acct_nums))}")

wallet_ids = [w["wallet_id"] for w in wallets]
print(f"  Unique wallet IDs?             {len(wallet_ids) == len(set(wallet_ids))}")

# Check timestamp uniqueness per account
ts_per_acct = defaultdict(list)
for t in transactions:
    ts_per_acct[t["customer_account_number"]].append((t["datestamp"], t["timestamp"]))

dups = 0
for acct_num, ts_list in ts_per_acct.items():
    c = Counter(ts_list)
    dups += sum(v - 1 for v in c.values() if v > 1)
print(f"  Duplicate timestamps per acct: {dups} (should be 0)")

print("\nNotebook 1 complete -- ready for typology injection")

